# Learning Graph Attention Networks for Anti-Money Laundering

**Goal:** learn how a transaction table becomes a graph, how a Graph Attention Network (GAT) creates account embeddings, and how those embeddings support AML account-risk classification.

This notebook is designed to run from the project folder alongside `dataset.py`, `model.py`, and `train.py`. It downloads the data from Kaggle through KaggleHub; it does not ask you to upload the large CSV manually.

> Educational baseline only. A predicted suspicious account is a model alert, not proof of money laundering.

## 0. Setup

Install the packages once. For GPU training, install the PyTorch build matching your CUDA version from the official PyTorch installation selector, then install a compatible PyTorch Geometric build.

Kaggle authentication is also required. Follow KaggleHub's current authentication instructions and keep API credentials out of Git.

In [ ]:
# If needed, uncomment and run:
# %pip install -r requirements.txt

import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt

from dataset import download_kaggle_dataset, locate_transaction_csv, load_or_build_graph
from model import AMLGAT
from train import make_splits, metrics_from_logits
from torch_geometric.loader import NeighborLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Download and inspect the Kaggle data

The default choice is `HI-Small_Trans.csv`, which is a practical starting point for learning. The Kaggle dataset may include larger variants. Change `PREFERRED_FILE` if you have the corresponding file in the downloaded dataset directory and enough memory.

The project uses KaggleHub so the notebook can download directly from the Kaggle dataset page.

In [ ]:
DATASET_DIR = download_kaggle_dataset()
PREFERRED_FILE = "HI-Small_Trans.csv"
csv_path = locate_transaction_csv(DATASET_DIR, PREFERRED_FILE)

print("Downloaded dataset directory:", DATASET_DIR)
print("Selected transaction file:", csv_path)

# Read only a small preview; do not load the full CSV just to inspect it.
preview = pd.read_csv(csv_path, nrows=8)
display(preview)
print("Columns:", preview.columns.tolist())

## 2. From transactions to a graph

Think of the raw data as an edge list:

- `From Bank` + `Account` identifies the sending account.
- `To Bank` + `Account.1` identifies the receiving account.
- One transaction becomes one directed edge: sender → receiver.
- The same account can appear in many transactions, so it is represented by one node.

We include the bank code in the account key because account numbers need not be globally unique across banks.

### What is the prediction target?

The supplied reference notebook turns transaction labels into account labels: if an account is either endpoint of at least one transaction with `Is Laundering = 1`, that account receives node label 1. This is a convenient node-classification learning task, but it is a proxy label, not a legal or investigative conclusion.

In [ ]:
# Build a PyG Data graph. CSV parsing is chunked to reduce raw-data peak memory.
# The resulting graph must still fit in RAM for this NeighborLoader example.
graph_path = "data/processed/aml_graph.pt"
data = load_or_build_graph(
    csv_path, graph_path, chunksize=250_000,
    force_rebuild=False, make_undirected=True,
)

print(data)
print(f"Accounts (nodes): {data.num_nodes:,}")
print(f"Transaction edges after reverse-edge option: {data.edge_index.size(1):,}")
print("Node feature shape:", tuple(data.x.shape))
print("Edge feature shape:", tuple(data.edge_attr.shape))
print("Positive account labels:", int(data.y.sum()), "/", data.num_nodes)
print("Positive rate:", f"{100 * data.y.float().mean().item():.5f}%")

## 3. Understand the PyG graph object

A PyTorch Geometric `Data` object stores the graph's tensors:

| Attribute | Shape | Meaning |
|---|---:|---|
| `x` | `[number_of_nodes, node_features]` | Account feature matrix |
| `edge_index` | `[2, number_of_edges]` | Source and destination node IDs |
| `edge_attr` | `[number_of_edges, edge_features]` | Transaction attributes for each edge |
| `y` | `[number_of_nodes]` | Account-level binary target |

`edge_index[0, e]` is the source node and `edge_index[1, e]` is the destination node for edge `e`.

By default, the builder adds the reverse of each transaction edge. This allows information to travel in both directions during message passing. It does **not** mean that the real-world transaction itself happened in both directions.

In [ ]:
print("First 5 edges (source -> destination):")
print(data.edge_index[:, :5])
print("Features for those edges:")
print(data.edge_attr[:5])
print("Features for first 5 accounts:")
print(data.x[:5])
print("Labels for first 20 accounts:")
print(data.y[:20].int())

## 4. What features does the graph contain?

**Node features** summarize account behavior:

- scaled bank identifier
- log of mean amount paid in each currency
- log of mean amount received in each currency

**Edge features** describe the transaction:

- normalized timestamp
- log amount paid and log amount received
- one-hot receiving currency, payment currency, and payment format

Log transforms reduce the influence of extremely large transaction values. One-hot encoding avoids pretending that category IDs have a meaningful numeric order.

The node aggregates are calculated over the entire selected CSV. That is acceptable for a first transductive exercise, but it can leak future information in a real prospective AML setting. A later section explains the more rigorous alternative.

In [ ]:
metadata = json.loads(data.metadata_json)
print(json.dumps(metadata, indent=2))
print("\nNode feature names:")
for name in data.feature_notes["node_features"]:
    print(" -", name)
print("\nEdge feature names:")
for name in data.feature_notes["edge_features"]:
    print(" -", name)

## 5. GAT intuition: learn which neighbors matter

A vanilla graph convolution often aggregates neighbor messages with a fixed rule. A Graph Attention Network learns a different importance weight for each neighbor.

For node `i`, a simplified attention operation is:

\[
e_{ij}=\operatorname{LeakyReLU}\left(
\mathbf a^T[\mathbf W h_i\,\Vert\,\mathbf W h_j\,\Vert\,\mathbf U e_{ij}^{edge}]
\right)
\]

\[
\alpha_{ij}=\operatorname{softmax}_{j\in\mathcal N(i)}(e_{ij}),
\qquad
h'_i=\sigma\left(\sum_{j\in\mathcal N(i)}\alpha_{ij}\mathbf W h_j\right)
\]

- `h_i`, `h_j`: current feature vectors for the target account and a neighboring account.
- `e_ij(edge)`: transaction features for the relationship.
- `alpha_ij`: learned attention weight, normalized across the target's neighborhood.
- `W`, `U`, and `a`: learned parameters.

This project uses `GATConv(edge_dim=...)`, so transaction attributes can influence attention. Multi-head attention repeats this calculation several times; concatenating heads gives the next layer a richer representation.

### Architecture in this project

1. **Input:** raw engineered account features.
2. **GAT layer 1:** 4 attention heads, each with 32 output channels. Concatenation produces 128 channels.
3. **ELU + dropout:** non-linearity and regularization.
4. **GAT layer 2:** combines neighborhood information into a 32-dimensional account embedding.
5. **Linear classifier:** maps each embedding to one logit (a raw score).
6. **Sigmoid at evaluation:** converts a logit to a probability between 0 and 1.

The training loss is `BCEWithLogitsLoss`, which combines sigmoid and binary cross-entropy in a numerically stable way. Do not apply sigmoid before passing logits to this loss.

In [ ]:
model = AMLGAT(
    in_channels=data.x.size(1),
    edge_dim=data.edge_attr.size(1),
    hidden_channels=32,
    heads=4,
    dropout=0.3,
)
print(model)

# A forward pass on the complete graph is useful for understanding tensor shapes.
# It may use substantial memory on a large graph.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
data = data.to(device)
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
print("Output logits shape:", tuple(logits.shape))  # one logit per account
print("Example probabilities:", torch.sigmoid(logits[:10]).flatten().cpu())

## 6. Train/validation/test split

We split **nodes**, not edges, because the task is account-level classification. Only training nodes contribute to the supervised loss in an epoch. During message passing, the sampled neighborhood can include other nodes; this is a standard transductive setup.

The labels are highly imbalanced in AML. A model can achieve deceptively high accuracy by predicting every account as normal. We therefore monitor:

- **PR-AUC / Average Precision:** useful when positives are rare.
- **Recall:** how many positive accounts are found.
- **Precision:** how many flagged accounts are truly positive in this dataset's labels.
- **F1:** harmonic mean of precision and recall.
- **Confusion matrix:** counts of true/false positives and negatives.

The split below is random and stratified to keep the positive rate similar across splits. It is not a temporal deployment evaluation.

In [ ]:
# Move graph back to CPU for NeighborLoader sampling.
data = data.cpu()
data = make_splits(data, seed=42, val_size=0.15, test_size=0.15)
for split_name, mask in [
    ("train", data.train_mask), ("validation", data.val_mask), ("test", data.test_mask)
]:
    print(f"{split_name:10s}: {int(mask.sum()):,} nodes | "
          f"{100 * data.y[mask].mean().item():.5f}% positive")

## 7. Mini-batch training with neighbor sampling

For each batch of seed accounts, `NeighborLoader` samples up to `k` neighbors at each GAT layer. With two GAT layers and `num_neighbors=[15, 15]`, the model gathers a two-hop neighborhood.

The first `batch.batch_size` nodes are the seed nodes. Compute loss only on those nodes; the remaining nodes exist to provide neighborhood context. This is important: computing loss on every sampled node would repeatedly and unintentionally train on context nodes.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 15],
    batch_size=512,
    input_nodes=data.train_mask,
    shuffle=True,
)
val_loader = NeighborLoader(
    data,
    num_neighbors=[15, 15],
    batch_size=512,
    input_nodes=data.val_mask,
    shuffle=False,
)

model = AMLGAT(
    in_channels=data.x.size(1),
    edge_dim=data.edge_attr.size(1),
    hidden_channels=32,
    heads=4,
    dropout=0.3,
).to(device)

# Class weighting increases the cost of missing rare positive accounts.
n_positive = data.y[data.train_mask].sum().item()
n_negative = int(data.train_mask.sum()) - n_positive
pos_weight = torch.tensor([n_negative / max(n_positive, 1)], device=device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)

print("Positive-class weight:", pos_weight.item())

In [ ]:
@torch.no_grad()
def evaluate_loader(model, loader, device, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        n_seed = int(batch.batch_size)
        logits = model(batch.x, batch.edge_index, batch.edge_attr)[:n_seed]
        all_logits.append(logits.cpu())
        all_labels.append(batch.y[:n_seed].cpu())
    return metrics_from_logits(torch.cat(all_logits), torch.cat(all_labels), threshold)

EPOCHS = 10  # Increase after the first successful run.
history = []
best_val_ap = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, seen = 0.0, 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        n_seed = int(batch.batch_size)
        logits = model(batch.x, batch.edge_index, batch.edge_attr)[:n_seed]
        labels = batch.y[:n_seed].float().view(-1, 1)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        loss_sum += loss.item() * n_seed
        seen += n_seed

    val_metrics = evaluate_loader(model, val_loader, device)
    row = {"epoch": epoch, "train_loss": loss_sum / max(seen, 1), **val_metrics}
    history.append(row)
    print(
        f"Epoch {epoch:02d} | loss={row['train_loss']:.5f} | "
        f"val PR-AUC={row['average_precision_pr_auc']:.5f} | "
        f"precision={row['precision']:.4f} | recall={row['recall']:.4f} | "
        f"F1={row['f1']:.4f}"
    )
    if val_metrics["average_precision_pr_auc"] > best_val_ap:
        best_val_ap = val_metrics["average_precision_pr_auc"]
        torch.save(model.state_dict(), "best_gat_notebook.pt")

## 8. Plot learning curves and evaluate on the held-out test nodes

The threshold `0.5` is only a starting point. In a real alerting workflow, choose a threshold using validation data and a target investigation capacity or acceptable false-positive rate. Do not tune the threshold on the test set.

In [ ]:
history_df = pd.DataFrame(history)
display(history_df[["epoch", "train_loss", "average_precision_pr_auc", "precision", "recall", "f1"]])

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training loss")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["average_precision_pr_auc"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation PR-AUC")
plt.title("Validation PR-AUC")
plt.grid(True)
plt.show()

In [ ]:
# Restore best validation checkpoint and run test evaluation once.
model.load_state_dict(torch.load("best_gat_notebook.pt", map_location=device, weights_only=True))
test_loader = NeighborLoader(
    data,
    num_neighbors=[15, 15],
    batch_size=512,
    input_nodes=data.test_mask,
    shuffle=False,
)
test_metrics = evaluate_loader(model, test_loader, device, threshold=0.5)
print(json.dumps(test_metrics, indent=2))

## 9. How to reason about the result

A high PR-AUC means the model ranks known positive accounts above negative accounts more effectively than a random ranking. It does not guarantee that a chosen alert threshold is operationally useful.

Check:
- How many positive accounts are found (recall)?
- How many alerts are correct under the dataset labels (precision)?
- How many false positives would investigators need to review?
- Does performance hold across banks, currencies, and time periods?
- Does the model outperform simple baselines such as logistic regression, random forest, or an MLP using the same node features?

## 10. Limitations and next experiments

1. **Temporal leakage:** node features are aggregated over all transactions in the CSV. Build features using only history before a cutoff, then score future accounts/transactions.
2. **Label construction:** the node label is positive if the account is an endpoint of a flagged transaction. This is a weak proxy and can make the problem easier than discovering previously unknown laundering.
3. **Random split:** use a chronological split for realistic prospective evaluation.
4. **Graph structure:** compare directed edges with bidirectional edges; compare GAT with GraphSAGE and GCN.
5. **Feature ablation:** remove edge attributes, then remove node amount aggregates, and compare PR-AUC/recall.
6. **Threshold selection:** select an operating point on validation data, never the test data.
7. **Explainability:** inspect high-attention neighbors for flagged accounts. Attention weights can be useful diagnostics, but they are not automatically faithful explanations.
8. **Scalability:** chunked CSV processing reduces raw-data memory pressure, but the final in-memory graph is still large. For very large variants, use partitioned/distributed graph storage and sampling.

## 11. Run the command-line training script

The same training pipeline is available outside the notebook:

```bash
python train.py --epochs 30 --batch-size 1024 --neighbors 15
```

See `README.md` for Kaggle authentication, memory tips, and CLI options.